# Limpieza y Preparación de Datos — Tendencias de YouTube (EE. UU.)

Notebook que toma el CSV **crudo** (`USvideos_cc50_202101.csv`), aplica una limpieza
documentada y genera un dataset **listo para modelar**.

- **Variable objetivo (target):** `likes` (una de las 4 sugeridas por el enunciado).
- **Grano:** panel completo (se conservan las apariciones de un mismo video en días de
  tendencia distintos; solo se eliminan duplicados exactos y por `(video_id, trending_date)`).
- **Regla anti-fuga:** como se predice `likes`, se **excluye** `like_view_ratio`
  (`likes/views`) de las variables del modelo. Los ratios que no contienen `likes`
  (`dislike_view_ratio`, `comment_view_ratio`) sí son válidos.

Se parte siempre del CSV crudo, no de estados intermedios.

## Flujo
1. Setup y carga
2. Deduplicación (panel)
3. Fechas y `days_to_trend`
4. Consistencia: negativos y reglas de negocio
5. Nulos
5.5 **Checkpoint** intermedio y re-análisis de variables
6. Ingeniería de variables (sin leakage)
7. Selección de columnas y exportación

## 1. Setup y carga

Importamos librerías, resolvemos la ruta de datos con `pathlib` (funciona desde
`notebooks/` o desde la raíz) y cargamos el CSV crudo cruzándolo con el catálogo de
categorías (`US_category_id.json`) para obtener `category_name`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
plt.style.use("ggplot")
sns.set_palette("husl")
%matplotlib inline

# Resolver la raiz del repositorio buscando la carpeta 'dataset/'
ROOT = Path.cwd()
while not (ROOT / "dataset").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CSV_PATH = ROOT / "dataset" / "USvideos_cc50_202101.csv"
JSON_PATH = ROOT / "dataset" / "US_category_id.json"

# Rutas de salida
CLEAN_BASE_PATH = ROOT / "dataset" / "USvideos_clean_base.csv"   # checkpoint (post-limpieza)
CLEAN_MODEL_PATH = ROOT / "dataset" / "USvideos_clean.csv"        # dataset final (model-ready)

print("Raiz del repo:", ROOT)
print("CSV crudo:", CSV_PATH.exists(), "| JSON:", JSON_PATH.exists())

In [ ]:
# Carga del CSV crudo + merge con el catalogo de categorias
df = pd.read_csv(CSV_PATH)

with open(JSON_PATH, "r", encoding="utf-8") as f:
    categories_data = json.load(f)
category_mapping = {
    int(item["id"]): item["snippet"]["title"] for item in categories_data["items"]
}
df["category_name"] = df["category_id"].map(category_mapping)

# Registro para el reporte final y contadores de filas eliminadas por motivo
n_filas_inicial = len(df)
descartes = {}

print(f"Filas iniciales: {n_filas_inicial:,} | Columnas: {df.shape[1]}")
df.head()

## 2. Deduplicación (panel)

Eliminamos duplicados **exactos** (filas idénticas en todas las columnas) y duplicados
por `(video_id, trending_date)` (mismo video, mismo día = registro redundante),
conservando la primera aparición. **No** colapsamos las apariciones legítimas de un
mismo video en días de tendencia distintos: eso es parte de la naturaleza del panel.

In [ ]:
# 2.1 Duplicados exactos
antes = len(df)
df = df.drop_duplicates()
descartes["duplicados_exactos"] = antes - len(df)

# 2.2 Duplicados por (video_id, trending_date), conservando el primero
antes = len(df)
df = df.drop_duplicates(subset=["video_id", "trending_date"], keep="first")
descartes["duplicados_video_fecha"] = antes - len(df)

print(f"Eliminados por duplicado exacto: {descartes['duplicados_exactos']}")
print(f"Eliminados por (video_id, trending_date): {descartes['duplicados_video_fecha']}")
print(f"Filas restantes: {len(df):,}")

## 3. Fechas y `days_to_trend`

Parseamos `trending_date` (formato `YY.DD.MM`) y `publish_time`, y construimos la
variable derivada `days_to_trend` = días entre la publicación y la entrada en tendencia.
Luego diagnosticamos su distribución para separar dos cosas distintas:

- **Valores negativos** → datos imposibles (tendencia antes de publicar): se descartan.
- **Extremos positivos** → outliers reales (videos antiguos que reingresan a tendencia):
  no se borran; se tratarán con `log1p` y, si hace falta, un cap por percentil.

In [ ]:
# Parseo de fechas
df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m", errors="coerce")
# publish_time viene con zona horaria (UTC); la quitamos para poder restar con trending_date
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce", utc=True).dt.tz_localize(None)

# Variable derivada: dias desde la publicacion hasta la tendencia.
# Se compara SOLO por fecha (sin hora) para evitar falsos negativos, igual que en la EDA:
# un video publicado a las 15:00 y en tendencia ese mismo dia daria -1 si se restara con hora.
df["days_to_trend"] = (df["trending_date"].dt.normalize() - df["publish_time"].dt.normalize()).dt.days

print("Fechas de tendencia no parseables:", int(df["trending_date"].isna().sum()))
print("Fechas de publicacion no parseables:", int(df["publish_time"].isna().sum()))
print("\ndays_to_trend -> describe():")
print(df["days_to_trend"].describe())
print("\nNegativos (imposibles):", int((df["days_to_trend"] < 0).sum()))

In [ ]:
# Diagnostico de outliers de days_to_trend (antes de limpiar): IQR sobre valores >= 0
positivos = df.loc[df["days_to_trend"] >= 0, "days_to_trend"]
q1, q3 = positivos.quantile(0.25), positivos.quantile(0.75)
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
print(f"Q1={q1:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}  limite_sup={lim_sup:.0f}")
print(f"Extremos por encima del limite IQR: {int((positivos > lim_sup).sum())}")
print(f"Percentil 99 de days_to_trend: {positivos.quantile(0.99):.0f} dias")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(positivos[positivos <= 60], bins=60, ax=axes[0])
axes[0].set_title("days_to_trend (0 a 60 dias)")
axes[0].set_xlabel("dias hasta tendencia")
sns.boxplot(x=np.log1p(positivos), ax=axes[1], color="salmon")
axes[1].set_title("log1p(days_to_trend) - boxplot")
plt.tight_layout()
plt.show()

## 4. Consistencia: negativos y reglas de negocio

Primero descartamos los `days_to_trend` negativos (imposibles). Luego validamos las 5
reglas de negocio de `scripts/eda.py`, reportamos las violaciones por regla y
eliminamos las filas que las incumplan (datos inconsistentes).

In [ ]:
# 4.1 Descartar days_to_trend negativos (imposibles)
antes = len(df)
df = df[df["days_to_trend"] >= 0].copy()
descartes["days_to_trend_negativos"] = antes - len(df)
print(f"Eliminados por days_to_trend negativo: {descartes['days_to_trend_negativos']}")
print(f"Filas restantes: {len(df):,}")

In [ ]:
# 4.2 Validacion de las 5 reglas de negocio (mascaras de VIOLACION)
def a_bool(serie):
    if serie.dtype == bool:
        return serie
    return serie.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

comments_disabled = a_bool(df["comments_disabled"])
ratings_disabled = a_bool(df["ratings_disabled"])

violaciones = {
    "R1a: likes > views": df["likes"] > df["views"],
    "R1b: dislikes > views": df["dislikes"] > df["views"],
    "R2: trending < publish (days<0)": df["days_to_trend"] < 0,
    "R3: comment_count > views": df["comment_count"] > df["views"],
    "R4: comments_disabled pero comment_count>0": comments_disabled & (df["comment_count"] > 0),
    "R5: ratings_disabled pero likes/dislikes>0": ratings_disabled & ((df["likes"] > 0) | (df["dislikes"] > 0)),
}

resumen_reglas = pd.DataFrame({"n_violaciones": {k: int(v.sum()) for k, v in violaciones.items()}})
resumen_reglas["%_violaciones"] = (resumen_reglas["n_violaciones"] / len(df) * 100).round(3)
resumen_reglas

In [ ]:
# 4.3 Eliminar filas que violan al menos una regla
mascara_violacion = pd.concat(violaciones.values(), axis=1).any(axis=1)
antes = len(df)
df = df[~mascara_violacion].copy()
descartes["viola_reglas_negocio"] = antes - len(df)
print(f"Eliminados por violar alguna regla de negocio: {descartes['viola_reglas_negocio']}")
print(f"Filas restantes: {len(df):,}")

## 5. Nulos

El único campo con nulos es `description` (~1.4%). Como no se usará como texto directo,
rellenamos con cadena vacía y dejamos una bandera `desc_faltante` por si aporta señal.
Verificamos que no queden nulos en las columnas que se usarán para modelar.

In [ ]:
# Nulos antes del tratamiento
print("Nulos por columna (antes):")
print(df.isnull().sum()[df.isnull().sum() > 0])

# description: bandera + relleno con cadena vacia
df["desc_faltante"] = df["description"].isna()
df["description"] = df["description"].fillna("")

# Verificacion: columnas clave para modelar no deben tener nulos
cols_modelo = ["likes", "views", "dislikes", "comment_count", "days_to_trend", "category_name"]
print("\nNulos en columnas de modelado (deberian ser 0):")
print(df[cols_modelo].isnull().sum())

## 5.5 Checkpoint intermedio y re-análisis de variables

Punto de control entre la limpieza y la ingeniería de variables:

- **Persistimos el estado limpio** (`df_base`) y lo exportamos como CSV intermedio
  (`USvideos_clean_base.csv`) para poder retomar el análisis/modelado sin re-ejecutar.
- **Verificamos** que la limpieza funcionó (0 duplicados, 0 nulos en columnas de
  modelado, `days_to_trend >= 0`).
- **Re-analizamos** sobre datos limpios: distribución saneada de `days_to_trend` y
  matriz de correlación **actualizada incluyendo `days_to_trend`** (Pearson sobre
  `log1p` y Spearman por la asimetría).

> Recordatorio: no se eliminan outliers de las métricas de engagement (son videos
> virales reales); se tratan con `log1p`. El borrado aplicó solo a lo imposible.

In [ ]:
# 5.5.1 Verificaciones de integridad post-limpieza
print("=== VERIFICACIONES POST-LIMPIEZA ===")
print("Duplicados exactos:", int(df.duplicated().sum()))
print("Duplicados (video_id, trending_date):", int(df.duplicated(subset=['video_id', 'trending_date']).sum()))
print("Nulos en columnas de modelado:", int(df[cols_modelo].isnull().sum().sum()))
print("days_to_trend negativos:", int((df['days_to_trend'] < 0).sum()))

# Persistir checkpoint en memoria y en disco
df_base = df.copy()
df_base.to_csv(CLEAN_BASE_PATH, index=False)
print(f"\nCheckpoint guardado en: {CLEAN_BASE_PATH}")
print(f"Filas en el checkpoint: {len(df_base):,}")

In [ ]:
# 5.5.2 Distribucion de days_to_trend ya saneada
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["days_to_trend"], bins=60, ax=axes[0])
axes[0].set_title("days_to_trend (limpio) - escala normal")
axes[0].set_xlabel("dias hasta tendencia")
sns.histplot(np.log1p(df["days_to_trend"]), bins=60, ax=axes[1], color="seagreen")
axes[1].set_title("log1p(days_to_trend)")
plt.tight_layout()
plt.show()

print(df["days_to_trend"].describe())

In [ ]:
# 5.5.3 Matriz de correlacion actualizada incluyendo days_to_trend
vars_corr = ["likes", "views", "dislikes", "comment_count", "days_to_trend"]
log_corr = np.log1p(df[vars_corr]).add_prefix("log_")

corr_pearson = log_corr.corr(method="pearson")
corr_spearman = df[vars_corr].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=axes[0])
axes[0].set_title("Pearson (log1p)")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title("Spearman (rangos)")
plt.tight_layout()
plt.show()

## 6. Ingeniería de variables

Construimos las variables derivadas para el modelo de `likes`:

- `first_trend` (bool): primera aparición de cada `video_id` en el panel.
- Ratios: `dislike_view_ratio`, `comment_view_ratio`.
- Transformaciones log: `log_likes` (target), `log_views`, `log_dislikes`,
  `log_comments`, `log_days_to_trend`.
- Derivadas de texto de bajo costo: `title_length`, `n_tags`.
- Codificación de `category_name` (códigos) para el modelo; se descarta `category_id`.

In [ ]:
# 6.1 first_trend: primera aparicion de cada video en el panel
primera_fecha = df.groupby("video_id")["trending_date"].transform("min")
df["first_trend"] = df["trending_date"].eq(primera_fecha)
print("Distribucion de first_trend:")
print(df["first_trend"].value_counts())

In [ ]:
# 6.2 Ratios validos (SIN 'likes' en el numerador para evitar leakage con el target)
# views > 0 en todo el dataset (min=549), pero usamos un guard por robustez.
denom = df["views"].replace(0, np.nan)
df["dislike_view_ratio"] = (df["dislikes"] / denom).fillna(0)
df["comment_view_ratio"] = (df["comment_count"] / denom).fillna(0)

# NOTA: like_view_ratio = likes/views se OMITE deliberadamente (leakage con target=likes).

# 6.3 Transformaciones logaritmicas (log_likes es el target)
df["log_likes"] = np.log1p(df["likes"])          # TARGET
df["log_views"] = np.log1p(df["views"])
df["log_dislikes"] = np.log1p(df["dislikes"])
df["log_comments"] = np.log1p(df["comment_count"])
df["log_days_to_trend"] = np.log1p(df["days_to_trend"])

# 6.4 Derivadas de texto de bajo costo (no leaky)
df["title_length"] = df["title"].astype(str).str.len()
df["n_tags"] = df["tags"].astype(str).apply(
    lambda t: 0 if t.strip() in ("", "[none]") else len(t.split("|"))
)

# 6.5 Codificacion de la categoria (codigos enteros) para el modelo
df["category_code"] = df["category_name"].astype("category").cat.codes

print("Nuevas variables creadas:")
print(df[["first_trend", "dislike_view_ratio", "comment_view_ratio",
          "log_likes", "log_views", "log_dislikes", "log_comments",
          "log_days_to_trend", "title_length", "n_tags", "category_code"]].head())

## 7. Selección de columnas y exportación

Descartamos columnas inservibles para modelar (identificadores únicos, geografía
aleatoria, textos crudos ya resumidos y columnas cuasi-constantes/redundantes),
conservamos identificadores solo para trazabilidad, y exportamos el dataset final
listo para modelar `likes`. Cerramos con un reporte de filas eliminadas por motivo.

In [ ]:
# 7.1 Definir columnas a conservar en el dataset model-ready
cols_id = ["video_id", "trending_date"]           # solo trazabilidad, NO features
col_target = ["likes", "log_likes"]                # target (crudo y log)
cols_features = [
    # numericas crudas (engagement, sin 'likes')
    "views", "dislikes", "comment_count", "days_to_trend",
    # transformaciones log
    "log_views", "log_dislikes", "log_comments", "log_days_to_trend",
    # ratios validos (no contienen likes)
    "dislike_view_ratio", "comment_view_ratio",
    # derivadas de texto / temporales / categoria
    "title_length", "n_tags", "first_trend", "desc_faltante",
    "category_name", "category_code",
    # banderas de estado del video
    "comments_disabled", "ratings_disabled",
]

cols_finales = cols_id + col_target + cols_features
df_model = df[cols_finales].copy()

# Columnas descartadas explicitamente (para el reporte)
cols_descartadas = [c for c in df.columns if c not in cols_finales]
print("Columnas conservadas:", len(cols_finales))
print("Columnas descartadas:", cols_descartadas)
df_model.head()

In [ ]:
# 7.2 Exportar dataset final (model-ready)
df_model.to_csv(CLEAN_MODEL_PATH, index=False)
print(f"Dataset model-ready guardado en: {CLEAN_MODEL_PATH}")
print(f"Dimensiones finales: {df_model.shape[0]:,} filas x {df_model.shape[1]} columnas")

In [ ]:
# 7.3 Reporte final de la limpieza
total_eliminadas = sum(descartes.values())
print("=== REPORTE DE LIMPIEZA ===")
print(f"Filas iniciales: {n_filas_inicial:,}")
for motivo, n in descartes.items():
    print(f"  - {motivo}: {n}")
print(f"Total eliminadas: {total_eliminadas}")
print(f"Filas finales: {len(df_model):,}  "
      f"({len(df_model) / n_filas_inicial * 100:.2f}% del original)")
print(f"\nTarget: likes (log_likes). Features: {len(cols_features)}. "
      f"Ratios con leakage excluidos: like_view_ratio.")